# 🎓 UTEQ - Laboratorio de Bromatología: Entrenamiento YOLOv8 y Exportación a TFLite
### Asistente Móvil Inteligente para la Detección de Equipos de Laboratorio

Este cuaderno permite entrenar un modelo **YOLOv8** personalizado para detectar los **15 equipos del Laboratorio de Bromatología de la UTEQ** y exportarlo directamente a formato **TensorFlow Lite (`.tflite`)** para su despliegue en la aplicación Android con **CameraX**.

## 1. Verificación del Entorno y GPU
Asegúrate de tener activada la aceleración por hardware por GPU en Colab: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4 o superior)`.

In [ ]:
!nvidia-smi

## 2. Instalación de Dependencias (Ultralytics y TensorFlow)

In [ ]:
!pip install -q ultralytics roboflow tensorflow matplotlib opencv-python

## 3. Carga del Dataset (Vía Google Drive o ZIP Directo)
Sube el archivo `dataset_bromatologia.zip` o monta tu Google Drive.

In [ ]:
# Opción A: Montar Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Opción B: Descomprimir dataset subido a Colab
import zipfile
import os

zip_path = 'dataset_bromatologia.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset_yolo')
    print('✅ Dataset descomprimido exitosamente!')
else:
    print('⚠️ Sube tu dataset_bromatologia.zip o vincula tu proyecto de Roboflow')

## 4. Crear Archivo de Configuración `data.yaml`

In [ ]:
data_yaml_content = """
path: /content/dataset_yolo
train: images/train
val: images/val
test: images/test

names:
  0: destilador_kjeldahl
  1: analizador_fibra
  2: campana_extraccion_gases
  3: estufa_secado_memmert
  4: molino_ciclonico_foss
  5: refractometro_atago
  6: calorimetro_bomba
  7: cabina_flujo_laminar_uvp
  8: sistema_tratamiento_agua
  9: destilador_agua
  10: bomba_vacio_recirculacion
  11: bomba_vacio_membrana
  12: gradilla_tubos_kjeldahl
  13: piseta_reactivo
  14: cilindro_gas
"""

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print('✅ data.yaml generado!')

## 5. Entrenamiento del Modelo YOLOv8 Nano (`yolov8n.pt`)
Entrenamos con tamaño de imagen 640x640 y data augmentation integrado para máxima robustez en el móvil.

In [ ]:
from ultralytics import YOLO

# Cargar modelo pre-entrenado YOLOv8 Nano
model = YOLO('yolov8n.pt')

# Entrenar
results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    save=True,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    augment=True,
    flipud=0.0,
    fliplr=0.5,
    degrees=10.0,
    scale=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)

## 6. Evaluación y Métricas de Rendimiento (mAP@50, Matriz de Confusión)

In [ ]:
# Validar mejor checkpoint
best_model = YOLO('runs/detect/train/weights/best.pt')
metrics = best_model.val(data='/content/data.yaml')

print(f"\n🎯 mAP@50: {metrics.box.map50:.4f}")
print(f"🎯 mAP@50-95: {metrics.box.map:.4f}")
print(f"🎯 Precision: {metrics.box.mp:.4f}")
print(f"🎯 Recall: {metrics.box.mr:.4f}")

In [ ]:
# Mostrar matriz de confusión y curvas de entrenamiento
from IPython.display import Image, display

display(Image(filename='runs/detect/train/confusion_matrix.png'))
display(Image(filename='runs/detect/train/results.png'))

## 7. Exportación a TensorFlow Lite (`.tflite` para Android)

In [ ]:
# Exportar a TFLite FP16 optimizado para inferencia en tiempo real en Android
tflite_path = best_model.export(format='tflite', imgsz=640, int8=False)
print(f'✅ Modelo exportado a: {tflite_path}')

## 8. Descarga del Modelo para la Aplicación Android

In [ ]:
from google.colab import files
import glob

tflite_files = glob.glob('runs/detect/train/weights/*_float16.tflite') + glob.glob('runs/detect/train/weights/*_float32.tflite') + glob.glob('runs/detect/train/weights/*.tflite')
if tflite_files:
    print(f'Descargando modelo: {tflite_files[0]}')
    files.download(tflite_files[0])
else:
    print('Busca el archivo en runs/detect/train/weights/best_saved_model/')